# 从 Tensor 到矩阵乘法

这一章只有一条主线：**把一批三维特征送进两层分类器，得到每条样本的两个类别分数**。为此，我们会先学会读 Tensor，再拆开点积、矩阵乘法和线性层，最后把前面写过的函数组装成 `TinyMLP`。

本章会反复使用同一批 `samples`、`weight` 和 `bias`。每个任务都在前一个任务的结果上前进，而不是换一组互不相关的练习。课程参考 PyTorch 官方的 [Tensors 教程](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)，以及 [Pytorch Matrix Multiplication](https://huggingface.co/spaces/not-lain/Pytorch-Matrix-Multiplication/tree/main) 对矩阵维度的拆解方式。

In [ ]:
import torch
from torch import nn
from typing import TypedDict

torch.manual_seed(7)

# 4 个样本，每个样本有 3 个输入特征
samples = torch.tensor([
    [1.0, 0.0, 2.0],
    [0.0, 1.0, 1.0],
    [2.0, 1.0, 0.0],
    [1.0, 2.0, 1.0],
], dtype=torch.float32)

# 第一层把 3 个特征变成 4 个隐藏特征
input_weight = torch.tensor([
    [0.5, -0.4, 0.8, 0.1],
    [0.3, 0.9, -0.2, 0.7],
    [-0.6, 0.2, 0.5, 0.4],
])
input_bias = torch.tensor([0.1, -0.2, 0.0, 0.3])

# 第二层把 4 个隐藏特征变成 2 个类别分数
output_weight = torch.tensor([
    [0.7, -0.5],
    [-0.3, 0.8],
    [0.6, 0.2],
    [0.1, 0.4],
])
output_bias = torch.tensor([0.05, -0.05])

## 1. 先读懂一个 Tensor

先给本章的数据一个明确的角色：我们要处理 **4 条传感器记录**，每条记录包含 **3 个数值特征**。把它们放进一个二维 Tensor 后，每一行是一条记录，每一列是同一种特征。这个 Tensor 叫作 `samples`，之后的选列、矩阵乘法和分类器都会继续使用它。

创建 Tensor 时，`torch.tensor(data, dtype=...)` 把 Python 数据变成 PyTorch 可以计算的对象。不要只记函数名；拿到一个 Tensor 后，先回答四个问题：

1. `shape`：每个维度分别有多大？
2. `ndim`：一共有几个维度？
3. `dtype`：每个元素用什么数值类型保存？
4. `device`：数据当前放在 CPU 还是其他设备？

下面的只读示例会创建本章贯穿始终的 `samples`，并打印这四项信息。你应当能从输出中读出：`(4, 3)` 表示 4 行 3 列，`ndim == 2` 表示它是二维表，`float32` 表示元素是 32 位浮点数，`cpu` 表示计算发生在 CPU。

### 查阅官方文档

- 创建数据：[`torch.tensor`](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)
- 阅读形状：[`Tensor.shape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.shape.html) 和 [`Tensor.ndim`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.ndim.html)
- 阅读存储信息：[`Tensor.dtype` 与 `Tensor.device`](https://docs.pytorch.org/docs/stable/tensor_attributes.html)

### 轮到你

完成 `tensor_summary(x)`，返回结构不是任意字典，而是具有以下固定契约的 `TensorSummary`：

| 固定键（`str`） | 对应值类型 | 示例值 |
| --- | --- | --- |
| `'shape'` | `tuple[int, ...]` | `(4, 3)` |
| `'ndim'` | `int` | `2` |
| `'dtype'` | `str` | `'torch.float32'` |
| `'device'` | `str` | `'cpu'` |

必须恰好返回这四个键，不要返回 `torch.Size`、`torch.dtype` 或 `torch.device` 对象。任务下方会立刻用这个函数检查贯穿本章的 `samples`。

In [ ]:
samples = torch.tensor([
    [1.0, 0.0, 2.0],
    [0.0, 1.0, 1.0],
    [2.0, 1.0, 0.0],
    [1.0, 2.0, 1.0],
], dtype=torch.float32)

print(samples)
print('shape :', samples.shape)   # torch.Size([4, 3])
print('ndim  :', samples.ndim)    # 2
print('dtype :', samples.dtype)   # torch.float32
print('device:', samples.device)  # cpu

In [ ]:
class TensorSummary(TypedDict):
    """tensor_summary 的固定返回结构。"""

    shape: tuple[int, ...]
    ndim: int
    dtype: str
    device: str


def tensor_summary(x: torch.Tensor) -> TensorSummary:
    """把任意 Tensor 的四项元数据整理成固定结构。

    Args:
        x: 任意 shape、dtype 和 device 的 torch.Tensor。

    Returns:
        恰好包含 shape、ndim、dtype、device 四个键的 TensorSummary。
        shape 是 tuple[int, ...]，ndim 是 int；dtype 和 device 是 str。

    Example:
        tensor_summary(torch.zeros(2, 3)) == {
            'shape': (2, 3),
            'ndim': 2,
            'dtype': 'torch.float32',
            'device': 'cpu',
        }
    """
    # TODO: 从 x 的属性中整理四项信息
    # 提示：shape 用 tuple(x.shape)，dtype/device 用 str(...)。
    pass

sample_info = tensor_summary(samples)
sample_info

## 2. 行是样本，列是特征

现在我们知道 `samples` 是一张 `[4, 3]` 的表。索引表达式 `x[行, 列]` 不是随意的语法：逗号左边决定保留哪些样本，右边决定保留哪些特征。

只写一个整数会消去对应维度。例如 `samples[0]` 取出第一条样本，形状从 `[4, 3]` 变成 `[3]`。切片 `:` 则表示该维全部保留，所以 `samples[:, 0]` 会取出 4 条样本的第一个特征。要一次选择多列，可以传入整数列表：`samples[:, [0, 2]]`。

观察只读示例里的每个结果及其 shape。尤其注意：`[0, 2]` 是特征编号，不是新数据；它让列数从 3 变成 2，但行数仍是 4。

### 查阅官方文档

- 基础索引与切片：[PyTorch Tensors 教程](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html#operations-on-tensors)
- 更显式的函数写法：[`torch.index_select`](https://docs.pytorch.org/docs/stable/generated/torch.index_select.html)

### 轮到你

完成 `select_features(x, columns)`。`x` 是 `[rows, features]` 的二维 Tensor，`columns` 是合法的零基列号列表。返回 `[rows, len(columns)]`，保持列号的顺序和重复项，并且不要原地修改 `x`。例如 `[2, 0, 2]` 表示依次取第 2、0、2 列。任务结尾会从本章 `samples` 中选出第 0、2 个特征，供后续观察。

In [ ]:
first_sample = samples[0]
first_feature = samples[:, 0]
two_features = samples[:, [0, 2]]

print(first_sample, first_sample.shape)      # [3]
print(first_feature, first_feature.shape)    # [4]
print(two_features, two_features.shape)      # [4, 2]

In [ ]:
def select_features(x: torch.Tensor, columns: list[int]) -> torch.Tensor:
    """按 columns 的顺序选择二维 Tensor 的特征列。

    Args:
        x: shape 为 [rows, features] 的二维 Tensor。
        columns: 合法的零基列号；顺序和重复项都必须保留。

    Returns:
        shape 为 [rows, len(columns)] 的 Tensor，dtype/device 与 x 相同。
        函数不得原地修改 x。

    Example:
        [[1, 2, 3], [4, 5, 6]] 选择 [2, 0]
        得到 [[3, 1], [6, 4]]。
    """
    # TODO: 使用二维索引返回选择结果
    pass

chosen_features = select_features(samples, [0, 2])
chosen_features

## 3. 点积：一行与一列如何产生一个数

下一步要让一条样本与一组权重发生计算。取 `samples` 的第一行 `[1, 0, 2]`，再取第一层权重的第一列 `[0.5, 0.3, -0.6]`。它们都是长度为 3 的向量，正好可以做点积。

点积分两步：先逐位置相乘，再把三个乘积相加。这里会得到 `[0.5, 0.0, -1.2]`，求和后是 `-0.7`。因此点积把“一个样本与一个输出特征的权重”压缩成一个数。矩阵乘法输出里的每个元素，本质上都来自这样一次点积。

在 PyTorch 中，`left * right` 或 `torch.mul(left, right)` 只做第一步，结果仍是向量；再调用 `.sum()` 才得到零维标量 Tensor。不要把逐元素乘法 `*` 和矩阵乘法 `@` 混在一起。

### 查阅官方文档

- 逐元素相乘：[`torch.mul`](https://docs.pytorch.org/docs/stable/generated/torch.mul.html)
- 沿维度或对全部元素求和：[`Tensor.sum`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.sum.html)
- 一维向量的直接点积：[`torch.dot`](https://docs.pytorch.org/docs/stable/generated/torch.dot.html)

### 轮到你

完成 `dot_product(left, right)`。两者必须是 shape 相同的一维 Tensor；返回值是零维标量 Tensor，并保持输入的 dtype/device。请明确写出“逐元素相乘，再求和”这两步，而不是调用 `torch.dot`，也不要把向量长度写死。任务结尾会计算第一条 `samples` 到第一个隐藏特征的值。

In [ ]:
sample_row = samples[0]
weight_column = input_weight[:, 0]
products = sample_row * weight_column
score = products.sum()

print(products)  # tensor([ 0.5000, 0.0000, -1.2000])
print(score)     # tensor(-0.7000)

In [ ]:
def dot_product(left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
    """把两个等长的一维 Tensor 做点积。

    Args:
        left: shape 为 [features] 的一维 Tensor。
        right: 与 left shape、dtype、device 相同的一维 Tensor。

    Returns:
        零维标量 Tensor，dtype/device 与输入相同。

    Example:
        [1, 2, 3] · [4, -1, 2] = 1*4 + 2*(-1) + 3*2 = 8。
    """
    # TODO: 逐元素相乘，再把所有乘积相加
    pass

first_hidden_value = dot_product(samples[0], input_weight[:, 0])
first_hidden_value

## 4. 矩阵乘法是一张行列点积表

上一节只计算了“一行样本 × 一列权重”。如果把 4 行样本依次与 4 列权重做点积，就会得到一张 `[4, 4]` 的结果表：4 行仍对应 4 条样本，4 列对应 4 个新特征。这就是二维矩阵乘法。

```text
[M, K] @ [K, N] -> [M, N]
输出[i, j] = dot(left[i, :], right[:, j])
```

中间维 `K` 必须一致，因为一次点积要求两个向量等长。`samples` 是 `[4, 3]`，`input_weight` 是 `[3, 4]`，共享的 3 正是每条样本的特征数，因此输出是 `[4, 4]`。

只读示例先用 PyTorch 的 `@` 算出整张表，再单独重算左上角元素。两个结果相等，说明矩阵乘法不是新的魔法操作，只是系统地枚举行列点积。

### 查阅官方文档

- 创建尚未初始化的输出：[`torch.empty`](https://docs.pytorch.org/docs/stable/generated/torch.empty.html)
- PyTorch 的通用矩阵乘法：[`torch.matmul`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)

`torch.empty` 只分配内存，不会自动填零，所以任务中必须写完输出的每个位置。创建输出时沿用 `left.dtype` 和 `left.device`，可以避免数据类型或 CPU/GPU 位置不一致。

### 轮到你

完成只处理二维输入的 `manual_matmul(left, right)`。输入分别为 `[M, K]` 和 `[K, N]`，且 dtype/device 相同；输出必须是 `[M, N]`，并沿用相同 dtype/device。创建输出后，用双重循环枚举 `row, column`，每个位置都调用上一节的 `dot_product(left[row, :], right[:, column])`，最后返回填完的 Tensor。

In [ ]:
matrix_result = samples @ input_weight
top_left = (samples[0] * input_weight[:, 0]).sum()

print(matrix_result.shape)       # torch.Size([4, 4])
print(matrix_result[0, 0])       # tensor(-0.7000)
print(top_left)                  # tensor(-0.7000)

In [ ]:
def manual_matmul(left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
    """用上一题的 dot_product 手写二维矩阵乘法。

    Args:
        left: shape 为 [M, K] 的二维 Tensor。
        right: shape 为 [K, N]，且 dtype/device 与 left 相同。

    Returns:
        shape 为 [M, N]、dtype/device 与 left 相同的 Tensor。
        output[row, column] 必须来自一次行列 dot_product。

    Example:
        [2, 3] @ [3, 4] 的输出 shape 是 [2, 4]。
    """
    rows, inner = left.shape
    inner_right, columns = right.shape
    assert inner == inner_right, "矩阵乘法的中间维度必须相同"
    output = torch.empty((rows, columns), dtype=left.dtype, device=left.device)
    # TODO: 用双重循环和 dot_product 填写 output[row, column]
    pass

hidden_without_bias = manual_matmul(samples, input_weight)
hidden_without_bias

## 5. 矩阵乘法加上 bias，就是线性层

矩阵乘法已经把每条样本的 3 个输入特征组合成 4 个新特征。神经网络的线性层还会给每个输出特征加一个可学习的起始偏移量 `bias`：

```text
samples [4, 3] @ weight [3, 4] + bias [4] -> hidden [4, 4]
```

这里最值得观察的是 `bias` 的 shape。它只有 `[4]`，并没有为 4 条样本各保存一份。PyTorch 会从最后一维对齐，把同一个 bias 自动加到结果的每一行，这种规则叫作 **broadcasting（广播）**。只读示例把第一行在加 bias 前后的数值并排打印出来，你可以直接看到每个位置恰好增加了对应的 bias。

### 查阅官方文档

- 矩阵乘法：[`torch.matmul`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)
- 自动扩展 bias：[Broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- 官方线性层函数：[`torch.nn.functional.linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.linear.html)

`torch.nn.functional.linear` 使用 `[out, in]` 排列 weight；本课为了让表达式直接写成 `x @ weight`，使用 `[in, out]`。两种约定都合理，但不能混用。

### 轮到你

完成 `linear_transform(x, weight, bias)`。`x` 为 `[batch, in_features]`，`weight` 为 `[in_features, out_features]`，`bias` 为 `[out_features]`；三者 dtype/device 相同。返回 `[batch, out_features]`，不修改任何输入。实现只需把两个操作连起来：先 `@`，再利用广播把同一个 bias 加到每一行。

In [ ]:
without_bias = samples @ input_weight
with_bias = without_bias + input_bias

print(without_bias[0])
print(input_bias)
print(with_bias[0])
print(with_bias.shape)  # torch.Size([4, 4])

In [ ]:
def linear_transform(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor,
) -> torch.Tensor:
    """计算二维批量数据的 x @ weight + bias。

    Args:
        x: shape 为 [batch, in_features] 的二维 Tensor。
        weight: shape 为 [in_features, out_features] 的二维 Tensor。
        bias: shape 为 [out_features] 的一维 Tensor。

    Returns:
        shape 为 [batch, out_features] 的 Tensor；dtype/device 与输入相同。
        不得原地修改 x、weight 或 bias。

    Example:
        [5, 3] @ [3, 4] + [4] -> [5, 4]。
    """
    # TODO: 使用 @ 完成矩阵乘法，再加上 bias
    pass

hidden = linear_transform(samples, input_weight, input_bias)
hidden

## 6. 更高维 Tensor：只替换最后一个特征维

到目前为止，每条样本只有一组特征，输入 shape 是 `[batch, features]`。序列模型会在中间多一个时间或 token 维度，输入变成 `[batch, sequence, features]`。线性变换仍然只关心最后的特征维：

```text
[B, T, C] @ [C, D] + [D] -> [B, T, D]
```

前面的 `B, T` 原样保留，最后的 `C` 被 weight 转换成 `D`。本章的数据共有 12 个元素，可以用 `reshape(2, 2, 3)` 重新解释为 2 组、每组 2 条记录、每条仍有 3 个特征。`reshape` 改变观察数据的方式，但不会凭空增加或删除元素。

只读示例表明，同一个 `x @ weight + bias` 表达式可以同时处理二维和三维输入。这种只约束最后一个特征维的思路，也为下一章批量计算字符的 Bigram logits 打下基础。

### 查阅官方文档

- 改变观察形状：[`Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)
- 高维矩阵乘法：[`torch.matmul`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)
- 两个三维 batch 矩阵的专用操作：[`torch.bmm`](https://docs.pytorch.org/docs/stable/generated/torch.bmm.html)

### 轮到你

完成 `batched_linear(x, weight, bias)`，并直接复用上一节的 `linear_transform`。`x` 可以是 `[B, C]` 或 `[B, T, C]`，`weight` 是 `[C, D]`，`bias` 是 `[D]`；输出分别为 `[B, D]` 或 `[B, T, D]`。不要根据 `x.ndim` 写分支，也不要写死 B、T、C、D。

In [ ]:
sequences = samples.reshape(2, 2, 3)
projected = sequences @ input_weight + input_bias

print(samples.shape)     # torch.Size([4, 3])
print(sequences.shape)   # torch.Size([2, 2, 3])
print(projected.shape)   # torch.Size([2, 2, 4])

In [ ]:
def batched_linear(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor,
) -> torch.Tensor:
    """复用 linear_transform，只替换 x 的最后一个特征维。

    Args:
        x: shape 为 [B, C] 或 [B, T, C] 的 Tensor。
        weight: shape 为 [C, D] 的二维 Tensor。
        bias: shape 为 [D] 的一维 Tensor。

    Returns:
        二维输入返回 [B, D]，三维输入返回 [B, T, D]；
        dtype/device 与输入相同。实现中不得按 ndim 分支。

    Example:
        [2, 5, 3] @ [3, 4] + [4] -> [2, 5, 4]。
    """
    # TODO: 复用 linear_transform；同一表达式应同时支持 2D 和 3D
    pass

sequences = samples.reshape(2, 2, 3)
sequence_hidden = batched_linear(sequences, input_weight, input_bias)
tensor_summary(sequence_hidden)

## 7. 把前面的运算组装成 TinyMLP

前六节已经准备好了完整模型所需的运算。现在只做组合：第一层把 3 个输入特征变成 4 个隐藏特征，`ReLU` 把负数截为 0，第二层再把 4 个隐藏特征变成 2 个类别分数。

```text
samples [4, 3]
  -> linear [4, 4]
  -> ReLU   [4, 4]
  -> linear [4, 2]  logits
```

PyTorch 用 `nn.Module` 表示一组可以复用、可以持有参数的计算。子类先在 `__init__` 中登记组件和参数，再在 `forward` 中描述数据如何流动。调用 `model(x)` 时，不要手动调用 `forward`；`Module` 会替你转发，并保留 hooks 等框架能力。

普通 Tensor 赋给属性后只是普通数据；包装成 `nn.Parameter` 后，`Module` 才会把它登记为可训练参数。只读示例实现了一个单层 `LinearBlock`：它还不是最终答案，但会让你看到 `named_parameters()` 能找到 weight 和 bias，以及 `ReLU` 不改变 shape。

### 查阅官方文档

- 模型基类：[`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
- 可训练参数：[`nn.Parameter`](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html)
- 非线性激活：[`torch.nn.functional.relu`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)
- 从类别分数得到索引：[`Tensor.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.argmax.html)

### 轮到你

完成 `TinyMLP`。四个构造参数的 shape 依次是 `[3, 4]`、`[4]`、`[4, 2]`、`[2]`；在 `__init__` 中分别 clone 后登记为同名的 `nn.Parameter`：`first_weight`、`first_bias`、`second_weight`、`second_bias`。`forward` 接受 `[batch, 3]`，复用 `linear_transform` 依次执行 `linear → ReLU → linear`，返回 `[batch, 2]` 的原始 logits，不要加 softmax。

In [ ]:
class LinearBlock(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: torch.Tensor) -> None:
        super().__init__()
        self.weight = nn.Parameter(weight.clone())
        self.bias = nn.Parameter(bias.clone())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(x @ self.weight + self.bias)

block = LinearBlock(input_weight, input_bias)
print([name for name, _ in block.named_parameters()])
# ['weight', 'bias']
print(block(samples).shape)  # torch.Size([4, 4])

In [ ]:
class TinyMLP(nn.Module):
    """把前六题的运算组装成一个 3 -> 4 -> 2 分类器。

    必须登记四个同名 nn.Parameter：first_weight [3, 4]、
    first_bias [4]、second_weight [4, 2]、second_bias [2]。
    """

    def __init__(
        self,
        first_weight: torch.Tensor,
        first_bias: torch.Tensor,
        second_weight: torch.Tensor,
        second_bias: torch.Tensor,
    ) -> None:
        super().__init__()
        # TODO: 分别 clone() 四个输入，再登记为同名 nn.Parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """计算未经 softmax 的两类 logits。

        Args:
            x: shape 为 [batch, 3] 的二维 Tensor。

        Returns:
            shape 为 [batch, 2] 的 Tensor。计算顺序必须是
            linear_transform -> torch.relu -> linear_transform。

        Example:
            [4, 3] 的输入返回 [4, 2] 的 logits。
        """
        # TODO: linear_transform -> torch.relu -> linear_transform
        pass

classifier = TinyMLP(input_weight, input_bias, output_weight, output_bias)
logits = classifier(samples)

## 最终运行：从输入 Tensor 得到分类结果

运行最后一个 Cell，查看每一步的形状和每个样本的预测。到这里，你写出的已经不是一道孤立的矩阵题，而是一个完整前向计算系统。下一课会继续使用同样的矩阵乘法规则：把字符索引编码成 one-hot 行向量，再乘上一张 `[V, V]` 权重表，得到下一个字符的 logits。

In [ ]:
logits = classifier(samples)
predictions = logits.argmax(dim=-1)
print('samples:    ', tuple(samples.shape))
print('hidden:     ', tuple(torch.relu(linear_transform(samples, input_weight, input_bias)).shape))
print('logits:     ', tuple(logits.shape))
print('predictions:', predictions.tolist())